# Week 2 — Counting Is Already a Model

**What you'll do today.** You'll build a word-counter by hand-logic, watch the same text get
counted three different ways, scale your count up with tf-idf, run the same counter across three
very different corpora, find one artist's *signature vocabulary*, and finish by counting an
*image*. The lesson underneath all of it: **counting requires defining**, and every definition
is a choice.

> If a code cell errors, don't panic. Read the last line, paste the whole error back to your AI
> with *"this errored, fix it and tell me what went wrong,"* and check
> `../kits/common-errors-cheatsheet.md`.

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib pillow

In [ ]:
# Imports. If this cell fails, it almost always means the install cell above didn't run
# this session — re-run it, then re-run this. (See the cheat sheet, entry 5.)
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
print("imports ok")

In [ ]:
# Are we running tiny + offline (the compatibility harness sets this), or in a real
# Colab class session? Everything below adapts so the notebook runs either way.
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Three little corpora

A *corpus* is just "a pile of text you're studying." We'll start with the one you live in — a
slice of pop lyrics — then a subreddit slice, then a public-domain novel. (In class the AI pulls
these live; here we keep small built-in samples so the notebook always runs.)

In [ ]:
# Small, self-contained stand-ins so the notebook runs anywhere. In class, the AI
# replaces these with live pulls (a lyrics slice, a subreddit export, a Gutenberg novel).
lyrics = [
    "we were both young when I first saw you I close my eyes",
    "cause baby now we got bad blood you know it used to be mad love",
    "it's a cruel summer with you and I burning",
    "shake it off shake it off I shake it off",
    "you call me up again just to break me like a promise",
]
reddit = [
    "AITA for not going to my sister's wedding after she uninvited my partner",
    "AITA for telling my roommate the dishes are not going to wash themselves",
    "AITA for leaving work early when my boss kept moving the deadline",
    "update I talked to my sister and we are okay now thanks everyone",
]
novel = (
    "It is a truth universally acknowledged that a single man in possession of a good "
    "fortune must be in want of a wife. However little known the feelings or views of "
    "such a man may be on his first entering a neighbourhood this truth is so well fixed."
)
corpora = {"lyrics": lyrics, "reddit": reddit, "novel": [novel]}
for name, docs in corpora.items():
    print(f"{name:8s}: {len(docs)} documents")

## Bag-of-words by hand: counting *requires defining*

Before any library, count by hand-logic. To count words you must first decide **what a word
is**. Lowercase everything? Is `run` the same as `running`? Do you drop common words like *the*?
Every one of those is a choice, and the choice changes the answer. The arguments *are* the
lesson.

In [ ]:
def tokenize(text, fold_case=True, strip_punct=True):
    if fold_case:
        text = text.lower()
    if strip_punct:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if t]

all_lyrics = " ".join(lyrics)
counts = Counter(tokenize(all_lyrics))
print("Top words in the lyrics slice:")
for word, n in counts.most_common(8):
    print(f"  {word:8s} {n}")

In [ ]:
# A "stop word" list is itself a choice. Watch the top-words list change when we drop
# common function words — the same kind of decision as merging run/running.
STOP = {"i","we","you","it","the","a","and","to","of","in","so","be","on","my","me","now"}
counts_nostop = Counter(t for t in tokenize(all_lyrics) if t not in STOP)
print("Top words with stop-words removed:")
for word, n in counts_nostop.most_common(8):
    print(f"  {word:8s} {n}")

### What counts as a word? Tokens, not words

Models never see words — they see **tokens**, and *which* tokens is a design choice (the same
`run`/`running` argument, automated). Try the same sentence in two tokenizer playgrounds and
watch it shatter differently:
[OpenAI Tokenizer](https://platform.openai.com/tokenizer) · [Tiktokenizer](https://tiktokenizer.vercel.app/).

## tf-idf: "common here, rare overall"

Raw counts are dominated by words that are common *everywhere* (*the*, *you*). **tf-idf**
reweights each word by how distinctive it is to one document — common *here*, rare *across the
whole pile*. We let the AI scale the hand count; reading the result is the skill.

In [ ]:
docs = lyrics + reddit + [novel]
labels = ["lyric"]*len(lyrics) + ["reddit"]*len(reddit) + ["novel"]
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform(docs)
terms = np.array(vec.get_feature_names_out())

# The most distinctive term in each lyric line:
for i in range(len(lyrics)):
    row = X[i].toarray().ravel()
    top = terms[row.argmax()]
    print(f"lyric {i}: most distinctive word -> '{top}'")

## Cross-corpus counting: the corpus choice changes what "common" means

Same counter, three corpora. Notice how the *corpus* you choose decides what looks "normal."

In [ ]:
for name, dd in corpora.items():
    text = " ".join(dd)
    top = Counter(t for t in tokenize(text) if t not in STOP).most_common(5)
    print(f"{name:8s}:", ", ".join(f"{w}({n})" for w, n in top))

### Delight beat — a signature vocabulary

Counting alone can show you a *voice*: the words one source uses far more than a baseline. (No
caveats here — just enjoy what the humblest tool already does.)

In [ ]:
# Words the lyrics use much more than the "baseline" (reddit + novel), by simple rate ratio.
def rates(text):
    toks = tokenize(text)
    n = len(toks) or 1
    return {w: c/n for w, c in Counter(toks).items()}

sig = rates(" ".join(lyrics))
base = rates(" ".join(reddit) + " " + novel)
distinctive = sorted(((w, sig[w]/(base.get(w,0)+1e-6)) for w in sig if w not in STOP),
                     key=lambda kv: kv[1], reverse=True)[:6]
print("Lyrics' signature words vs. baseline:")
for w, r in distinctive:
    print(f"  {w}")

## Counting images: pictures are data too

A picture is numbers — pixels with red/green/blue values. So *counting* works on images exactly
like words: average color, brightness, a "darkest cover" ranking. And **what you choose to
count** (hue? brightness? saturation?) is the same kind of decision as stop-words. Image
projects start here and run all the way through the course.

In [ ]:
from PIL import Image

# In class these are real album covers fetched by the AI. Here we make a few solid-color
# swatches so the *counting* logic is identical and always runs.
def swatch(rgb, size=64):
    arr = np.zeros((size, size, 3), dtype=np.uint8)
    arr[:, :] = rgb
    return Image.fromarray(arr)

covers = {
    "midnight":  swatch((20, 20, 40)),
    "sunburst":  swatch((240, 180, 40)),
    "forest":    swatch((30, 90, 50)),
}

def avg_brightness(img):
    a = np.asarray(img).astype(float)
    # perceived luminance
    return float((0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).mean())

ranked = sorted(covers.items(), key=lambda kv: avg_brightness(kv[1]))
print("Darkest -> lightest album cover:")
for name, img in ranked:
    print(f"  {name:9s} brightness={avg_brightness(img):6.1f}")

In [ ]:
# A brightness "histogram" for one cover — the same move as a word-frequency bar chart.
a = np.asarray(covers["sunburst"]).astype(float)
lum = (0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).ravel()
plt.figure(figsize=(5,3))
plt.hist(lum, bins=20)
plt.title("Brightness histogram — 'sunburst' cover")
plt.xlabel("luminance"); plt.ylabel("pixels")
plt.tight_layout(); plt.show()

## You made a counter

You defined what a word is, counted three corpora three ways, found a voice with nothing but
counting, and counted a picture the same way. Next week the machine starts *weighting* those
counts — that's classification.

**Sketch (homework):** count something in a text you love; make one chart; write one sentence
naming a choice you made.